# 00b — Register Fabric OneLake as an AML datastore

Registers the partner's Fabric lakehouse as a first-class **AML datastore** so it appears in the AML Studio UI under **Data > Datastores** and can be referenced by `azureml://datastores/<name>/paths/...`.

This is complementary to the **copy-into-AML** pattern in `01a-copy-fabric-to-aml.ipynb`:

| Pattern | When to use |
|---------|-------------|
| **OneLake datastore** (this notebook) | Browse / reference Fabric data in place from AML; good for ad-hoc reads. |
| **Copy + MLTable data asset** (`01a`) | Versioned snapshot for training so AML tracks lineage; what we use for the POC training run. |

**Prereqs**
- The AML workspace identity (or your `az login` user) has at least **Viewer** on the Fabric workspace and **Read** on the lakehouse.
- `.env` contains `FABRIC_WORKSPACE_NAME` and `FABRIC_LAKEHOUSE_NAME`.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from src.register_onelake_datastore import register_onelake_datastore

name = register_onelake_datastore(datastore_name='fabric_onelake')
print('Datastore registered:', name)

## Verify by listing tables under the lakehouse path

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from src.config import load_settings

s = load_settings()
ml = MLClient(DefaultAzureCredential(), s.subscription_id, s.resource_group, s.aml_workspace)
ds = ml.datastores.get('fabric_onelake')
print('Datastore:', ds.name)
print('Workspace:', ds.one_lake_workspace_name)
print('Artifact: ', ds.artifact.name)

## Preview data from a OneLake Delta table

The lakehouse `Tables/` folder contains **Delta** tables. We can read one
directly from its `abfss://` URI using the `deltalake` library and an
Entra bearer token (scope `https://storage.azure.com/.default`) — no
datastore mount needed.

This proves end-to-end read access against the live Fabric data.

In [ ]:
from deltalake import DeltaTable

# OneLake URI built from .env (FABRIC_WORKSPACE_NAME, FABRIC_LAKEHOUSE_NAME,
# ONELAKE_SCHEMA, ONELAKE_TABLE)
table_uri = s.onelake_table_uri
print("Reading:", table_uri)

# deltalake needs a bearer token for the storage scope
token = DefaultAzureCredential().get_token("https://storage.azure.com/.default").token
storage_options = {"bearer_token": token, "use_fabric_endpoint": "true"}

dt = DeltaTable(table_uri, storage_options=storage_options)
print("Version:", dt.version())
print("Files:  ", len(dt.file_uris()))

df = dt.to_pandas()
print(f"\nShape: {df.shape}")
print("Columns:", list(df.columns))
df.head(10)